In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Output
from pathlib import Path

def plot_emissions_and_stocks(case_study, obj_mode, folder_path_stocks, folder_path_emissions):
    n_steps = 100
    fig_folder_path = Path('./plots/fig') / case_study
    fig_folder_path.mkdir(parents=True, exist_ok=True)

    output_file_path = fig_folder_path / f"{case_study}_{obj_mode}_emissions_stocks_scenarios_comparisons.svg"

    prefix = f"{case_study}_{obj_mode}"

    def custom_sort_key(file_name):
        order = {
            'bau': 0, 'no_cons': 1, 'AAC_90': 2, 'AAC_80': 3, 'AAC_70': 4,
            'AAC_60': 5, 'AAC_50': 6, 'AAC_40': 7, 'AAC_30': 8, 'AAC_20': 9, 'AAC_10': 10
        }
        for key, value in order.items():
            if key in file_name:
                return value
        return 11  # Default for any unlisted scenarios

    def load_and_sort_csv_files(folder_path, suffix):
        folder = Path(folder_path)
        csv_files = sorted(
            [f for f in folder.iterdir() if f.name.startswith(prefix) and f.name.endswith(suffix)],
            key=lambda x: custom_sort_key(x.name)
        )
        if not csv_files:
            raise ValueError(f"No files found matching the prefix '{prefix}' in the folder '{folder_path}'.")
        return [pd.read_csv(file).reset_index(drop=True) for file in csv_files]

    cbm_outputs_stocks = load_and_sort_csv_files(folder_path_stocks, 'cbm_output_1.csv')
    cbm_outputs_emissions = load_and_sort_csv_files(folder_path_emissions, 'cbm_output_2.csv')

    scenario_colors = ['black'] + ['blue', 'green', 'orange', 'purple', 'red', 'brown', 'pink', 'gray', 'cyan', 'magenta']

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    def plot_scenarios(ax, data, column, title, ylabel):
        for i, df in enumerate(data):
            label = 'Baseline' if i == 0 else f'S{i-1}'
            color = scenario_colors[i]
            df[column].plot(ax=ax, xlim=(0, n_steps), label=label, color=color)
        ax.set_title(title)
        ax.set_xlabel('Year')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True)

    plot_scenarios(
        axes[0], cbm_outputs_stocks, 'Total Ecosystem',
        title='Total Ecosystem Stocks Across Scenarios',
        ylabel='Total Ecosystem Stocks (tC)'
    )

    plot_scenarios(
        axes[1], cbm_outputs_emissions, 'Net emission',
        title='Net Emissions Across Scenarios',
        ylabel='Net Emissions (tCO₂e)'
    )
    axes[1].axhline(0, color='red', linestyle='--', linewidth=1)  

    plt.tight_layout()
    plt.savefig(output_file_path)
    plt.show()

# Interactive function
def interactive_plot_emissions_and_stocks(case_study, obj_mode):
    folder_path_stocks = f'./plots/{case_study}/stocks'
    folder_path_emissions = f'./plots/{case_study}/emissions'
    plot_emissions_and_stocks(case_study, obj_mode, folder_path_stocks, folder_path_emissions)

# Widgets for user input
case_study_dropdown = Dropdown(
    options=['equitysilver', 'goldenbear', 'redchris'],  # Add more case study options
    value='equitysilver',
    description='Case Study:'
)

obj_mode_dropdown = Dropdown(
    options=['max_hv', 'min_ha', 'max_st', 'min_em'],  # Add more objective modes
    value='max_hv',
    description='Objective Mode:'
)

# Display interactive widgets
interact(interactive_plot_emissions_and_stocks, case_study=case_study_dropdown, obj_mode=obj_mode_dropdown)


interactive(children=(Dropdown(description='Case Study:', options=('equitysilver', 'goldenbear', 'redchris'), …

<function __main__.interactive_plot_emissions_and_stocks(case_study, obj_mode)>

In [3]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Text, Output

def plot_emissions_and_stocks(case_study, obj_mode, folder_path_stocks, folder_path_emissions):
    # Constants
    n_steps = 100
    fig_folder_path = os.path.join('./plots/fig', case_study)
    if not os.path.exists(fig_folder_path):
        os.makedirs(fig_folder_path)
    
    # Custom sorting function for files
    def custom_sort_key(file_name):
        sort_order = [
            'bau', 'no_cons', 'AAC_90', 'AAC_80', 'AAC_70', 
            'AAC_60', 'AAC_50', 'AAC_40', 'AAC_30', 
            'AAC_20', 'AAC_10'
        ]
        for idx, keyword in enumerate(sort_order):
            if keyword in file_name:
                return idx
        return len(sort_order)

    # Helper function to load and sort files
    def load_and_sort_files(folder_path, prefix, suffix):
        files = [
            f for f in os.listdir(folder_path)
            if f.startswith(prefix) and f.endswith(suffix)
        ]
        if not files:
            raise ValueError(f"No files found matching the prefix '{prefix}' in the folder '{folder_path}'.")
        files.sort(key=custom_sort_key)
        return [pd.read_csv(os.path.join(folder_path, file)) for file in files]
    
    # Load stock and emission data
    prefix = f"{case_study}_{obj_mode}"
    cbm_outputs_stocks = load_and_sort_files(folder_path_stocks, prefix, 'cbm_output_1.csv')
    cbm_outputs_emissions = load_and_sort_files(folder_path_emissions, prefix, 'cbm_output_2.csv')

    # Ensure consistent indexing
    for df in cbm_outputs_stocks + cbm_outputs_emissions:
        df.index = range(1, len(df) + 1)

    # Plot Total Ecosystem Stocks
    fig1, ax1 = plt.subplots(figsize=(7, 6))
    scenario_colors = ['blue', 'green', 'orange', 'purple', 'red', 'brown', 'pink', 'gray', 'cyan', 'magenta']

    for i, df in enumerate(cbm_outputs_stocks):
        label = 'Baseline' if i == 0 else f'S{i-1}'
        color = 'black' if i == 0 else scenario_colors[i-1]
        df['Total Ecosystem'].plot(ax=ax1, xlim=(0, n_steps), label=label, color=color)

    ax1.set_title('Total Ecosystem Stocks Across Scenarios')
    ax1.set_xlabel('Year')
    ax1.set_ylabel('Total Ecosystem Stocks (tC)')
    ax1.legend()
    ax1.grid(True)
    
    # Save the first figure
    fig1_path = os.path.join(fig_folder_path, f"{case_study}_{obj_mode}_stocks_comparisons.svg")
    fig1.tight_layout()
    fig1.savefig(fig1_path)
    
    # Plot Net Emissions
    fig2, ax2 = plt.subplots(figsize=(7, 6))
    for i, df in enumerate(cbm_outputs_emissions):
        label = 'Baseline' if i == 0 else f'S{i-1}'
        color = 'black' if i == 0 else scenario_colors[i-1]
        df['Net emission'].plot(ax=ax2, xlim=(0, n_steps), label=label, color=color)

    # Add horizontal red line for y=0
    ax2.axhline(0, color='red', linestyle='--', linewidth=1)

    ax2.set_title('Net Emissions Across Scenarios')
    ax2.set_xlabel('Year')
    ax2.set_ylabel('Net Emissions (tCO₂e)')
    ax2.legend()
    ax2.grid(True)
    
    # Save the second figure
    fig2_path = os.path.join(fig_folder_path, f"{case_study}_{obj_mode}_emissions_comparisons.svg")
    fig2.tight_layout()
    fig2.savefig(fig2_path)
    
    # Show both figures
    plt.show()

# Interactive function
def interactive_plot_emissions_and_stocks(case_study, obj_mode):
    folder_path_stocks = f'./plots/{case_study}/stocks'
    folder_path_emissions = f'./plots/{case_study}/emissions'
    plot_emissions_and_stocks(case_study, obj_mode, folder_path_stocks, folder_path_emissions)

# Widgets for user input
case_study_dropdown = Dropdown(
    options=['equitysilver', 'goldenbear', 'redchris'],  # Add more case study options
    value='equitysilver',
    description='Case Study:'
)

obj_mode_dropdown = Dropdown(
    options=['max_hv', 'min_ha', 'max_st', 'min_em'],  # Add more objective modes
    value='max_hv',
    description='Objective Mode:'
)

# Display interactive widgets
interact(interactive_plot_emissions_and_stocks, case_study=case_study_dropdown, obj_mode=obj_mode_dropdown)


interactive(children=(Dropdown(description='Case Study:', options=('equitysilver', 'goldenbear', 'redchris'), …

<function __main__.interactive_plot_emissions_and_stocks(case_study, obj_mode)>

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Output

def plot_emissions_and_stocks(case_study, folder_path_stocks, folder_path_emissions):
    # Constants
    n_steps = 100
    fig_folder_path = os.path.join('./plots/fig', case_study)
    if not os.path.exists(fig_folder_path):
        os.makedirs(fig_folder_path)

    # Custom sorting function for files
    def custom_sort_key(file_name):
        sort_order = [
            'bau', 'no_cons', 'AAC_90', 'AAC_80', 'AAC_70', 
            'AAC_60', 'AAC_50', 'AAC_40', 'AAC_30', 
            'AAC_20', 'AAC_10'
        ]
        for idx, keyword in enumerate(sort_order):
            if keyword in file_name:
                return idx
        return len(sort_order)

    # Helper function to load and sort files
    def load_and_sort_files(folder_path, prefix, suffix):
        files = [
            f for f in os.listdir(folder_path)
            if f.startswith(prefix) and f.endswith(suffix)
        ]
        if not files:
            raise ValueError(f"No files found matching the prefix '{prefix}' in the folder '{folder_path}'.")
        files.sort(key=custom_sort_key)
        return [pd.read_csv(os.path.join(folder_path, file)) for file in files]

    # List of objective modes
    obj_modes = ['max_hv', 'min_ha', 'max_st', 'min_em']

    # Scenario colors for each objective mode
    scenario_colors = ['blue', 'green', 'orange', 'purple', 'red', 'brown', 'pink', 'gray', 'cyan', 'magenta']

    # Plot Total Ecosystem Stocks (One figure with 4 subplots)
    fig1, axes1 = plt.subplots(2, 2, figsize=(14, 10))
    for i, obj_mode in enumerate(obj_modes):
        prefix = f"{case_study}_{obj_mode}"
        
        # Load stock data
        cbm_outputs_stocks = load_and_sort_files(folder_path_stocks, prefix, 'cbm_output_1.csv')

        # Ensure consistent indexing
        for df in cbm_outputs_stocks:
            df.index = range(1, len(df) + 1)

        # Plot Total Ecosystem Stocks for each obj_mode
        ax = axes1[i // 2, i % 2]  # Select subplot position
        for j, df in enumerate(cbm_outputs_stocks):
            label = 'Baseline' if j == 0 else f'S{j-1}'
            color = 'black' if j == 0 else scenario_colors[j-1]
            df['Total Ecosystem'].plot(ax=ax, xlim=(0, n_steps), label=label, color=color)
        ax.set_title(f'Total Ecosystem Stocks ({obj_mode})')
        ax.set_xlabel('Year')
        ax.set_ylabel('Total Ecosystem Stocks (tC)')
        ax.legend()
        ax.grid(True)
    
    # Save the figure for Total Ecosystem Stocks
    fig1.tight_layout()
    fig1_path = os.path.join(fig_folder_path, f"{case_study}_total_ecosystem_stocks_comparisons.svg")
    fig1.savefig(fig1_path)

    # Plot Net Emissions (One figure with 4 subplots)
    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
    for i, obj_mode in enumerate(obj_modes):
        prefix = f"{case_study}_{obj_mode}"

        # Load emission data
        cbm_outputs_emissions = load_and_sort_files(folder_path_emissions, prefix, 'cbm_output_2.csv')

        # Ensure consistent indexing
        for df in cbm_outputs_emissions:
            df.index = range(1, len(df) + 1)

        # Plot Net Emissions for each obj_mode
        ax = axes2[i // 2, i % 2]  # Select subplot position
        for j, df in enumerate(cbm_outputs_emissions):
            label = 'Baseline' if j == 0 else f'S{j-1}'
            color = 'black' if j == 0 else scenario_colors[j-1]
            df['Net emission'].plot(ax=ax, xlim=(0, n_steps), label=label, color=color)
        ax.axhline(0, color='red', linestyle='--', linewidth=1)  # Horizontal line for zero emissions
        ax.set_title(f'Net Emissions ({obj_mode})')
        ax.set_xlabel('Year')
        ax.set_ylabel('Net Emissions (tCO₂e)')
        ax.legend()
        ax.grid(True)

    # Save the figure for Net Emissions
    fig2.tight_layout()
    fig2_path = os.path.join(fig_folder_path, f"{case_study}_net_emissions_comparisons.svg")
    fig2.savefig(fig2_path)

    # Show both figures
    plt.show()

# Interactive function
def interactive_plot_emissions_and_stocks(case_study):
    folder_path_stocks = f'./plots/{case_study}/stocks'
    folder_path_emissions = f'./plots/{case_study}/emissions'
    plot_emissions_and_stocks(case_study, folder_path_stocks, folder_path_emissions)

# Widgets for user input
case_study_dropdown = Dropdown(
    options=['equitysilver', 'goldenbear', 'redchris'],  # Add more case study options
    value='equitysilver',
    description='Case Study:'
)

# Display interactive widgets
interact(interactive_plot_emissions_and_stocks, case_study=case_study_dropdown)


interactive(children=(Dropdown(description='Case Study:', options=('equitysilver', 'goldenbear', 'redchris'), …

<function __main__.interactive_plot_emissions_and_stocks(case_study)>

In [5]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, Output

def plot_emissions_and_stocks(case_study, folder_path_stocks, folder_path_emissions):
    # Constants
    n_steps = 100
    fig_folder_path = os.path.join('./plots/fig', case_study)
    if not os.path.exists(fig_folder_path):
        os.makedirs(fig_folder_path)

    # Custom sorting function for files
    def custom_sort_key(file_name):
        sort_order = [
            'bau', 'no_cons', 'AAC_90', 'AAC_80', 'AAC_70', 
            'AAC_60', 'AAC_50', 'AAC_40', 'AAC_30', 
            'AAC_20', 'AAC_10'
        ]
        for idx, keyword in enumerate(sort_order):
            if keyword in file_name:
                return idx
        return len(sort_order)

    # Helper function to load and sort files
    def load_and_sort_files(folder_path, prefix, suffix):
        files = [
            f for f in os.listdir(folder_path)
            if f.startswith(prefix) and f.endswith(suffix)
        ]
        if not files:
            raise ValueError(f"No files found matching the prefix '{prefix}' in the folder '{folder_path}'.")
        files.sort(key=custom_sort_key)
        return [pd.read_csv(os.path.join(folder_path, file)) for file in files]

    # List of objective modes
    obj_modes = ['max_hv', 'min_ha', 'max_st', 'min_em']

    # Scenario colors for each objective mode
    scenario_colors = ['blue', 'green', 'orange', 'purple', 'red', 'brown', 'pink', 'gray', 'cyan', 'magenta']

    # Plot Total Ecosystem Stocks (One figure with 4 subplots)
    fig1, axes1 = plt.subplots(2, 2, figsize=(14, 10))
    ylims_stocks = [float('inf'), float('-inf')]  # Initialize min and max values for y-axis
    for i, obj_mode in enumerate(obj_modes):
        prefix = f"{case_study}_{obj_mode}"
        
        # Load stock data
        cbm_outputs_stocks = load_and_sort_files(folder_path_stocks, prefix, 'cbm_output_1.csv')

        # Ensure consistent indexing
        for df in cbm_outputs_stocks:
            df.index = range(1, len(df) + 1)

        # Plot Total Ecosystem Stocks for each obj_mode
        ax = axes1[i // 2, i % 2]  # Select subplot position
        for j, df in enumerate(cbm_outputs_stocks):
            label = 'Baseline' if j == 0 else f'S{j-1}'
            color = 'black' if j == 0 else scenario_colors[j-1]
            df['Total Ecosystem'].plot(ax=ax, xlim=(0, n_steps), label=label, color=color)
        ax.set_title(f'Total Ecosystem Stocks ({obj_mode})')
        ax.set_xlabel('Year')
        ax.set_ylabel('Total Ecosystem Stocks (tC)')
        ax.legend()
        ax.grid(True)

        # Track the min and max y values across all subplots for consistent y-axis scaling
        ylims_stocks[0] = min(ylims_stocks[0], ax.get_ylim()[0])
        ylims_stocks[1] = max(ylims_stocks[1], ax.get_ylim()[1])

    # Apply identical y-axis limits to all subplots for Total Ecosystem Stocks
    for ax in axes1.flat:
        ax.set_ylim(ylims_stocks)

    # Save the figure for Total Ecosystem Stocks
    fig1.tight_layout()
    fig1_path = os.path.join(fig_folder_path, f"{case_study}_total_ecosystem_stocks_comparisons.svg")
    fig1.savefig(fig1_path)

    # Plot Net Emissions (One figure with 4 subplots)
    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
    ylims_emissions = [float('inf'), float('-inf')]  # Initialize min and max values for y-axis
    for i, obj_mode in enumerate(obj_modes):
        prefix = f"{case_study}_{obj_mode}"

        # Load emission data
        cbm_outputs_emissions = load_and_sort_files(folder_path_emissions, prefix, 'cbm_output_2.csv')

        # Ensure consistent indexing
        for df in cbm_outputs_emissions:
            df.index = range(1, len(df) + 1)

        # Plot Net Emissions for each obj_mode
        ax = axes2[i // 2, i % 2]  # Select subplot position
        for j, df in enumerate(cbm_outputs_emissions):
            label = 'Baseline' if j == 0 else f'S{j-1}'
            color = 'black' if j == 0 else scenario_colors[j-1]
            df['Net emission'].plot(ax=ax, xlim=(0, n_steps), label=label, color=color)
        ax.axhline(0, color='red', linestyle='--', linewidth=1)  # Horizontal line for zero emissions
        ax.set_title(f'Net Emissions ({obj_mode})')
        ax.set_xlabel('Year')
        ax.set_ylabel('Net Emissions (tCO₂e)')
        ax.legend()
        ax.grid(True)

        # Track the min and max y values across all subplots for consistent y-axis scaling
        ylims_emissions[0] = min(ylims_emissions[0], ax.get_ylim()[0])
        ylims_emissions[1] = max(ylims_emissions[1], ax.get_ylim()[1])

    # Apply identical y-axis limits to all subplots for Net Emissions
    for ax in axes2.flat:
        ax.set_ylim(ylims_emissions)

    # Save the figure for Net Emissions
    fig2.tight_layout()
    fig2_path = os.path.join(fig_folder_path, f"{case_study}_net_emissions_comparisons.svg")
    fig2.savefig(fig2_path)

    # Show both figures
    plt.show()

# Interactive function
def interactive_plot_emissions_and_stocks(case_study):
    folder_path_stocks = f'./plots/{case_study}/stocks'
    folder_path_emissions = f'./plots/{case_study}/emissions'
    plot_emissions_and_stocks(case_study, folder_path_stocks, folder_path_emissions)

# Widgets for user input
case_study_dropdown = Dropdown(
    options=['equitysilver', 'goldenbear', 'redchris'],  # Add more case study options
    value='equitysilver',
    description='Case Study:'
)

# Display interactive widgets
interact(interactive_plot_emissions_and_stocks, case_study=case_study_dropdown)


interactive(children=(Dropdown(description='Case Study:', options=('equitysilver', 'goldenbear', 'redchris'), …

<function __main__.interactive_plot_emissions_and_stocks(case_study)>

In [6]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown

def plot_emissions_and_stocks(case_study, folder_path_stocks, folder_path_emissions):
    # Constants
    n_steps = 100
    fig_folder_path = os.path.join('./plots/fig', case_study)
    if not os.path.exists(fig_folder_path):
        os.makedirs(fig_folder_path)

    # Custom sorting function for files
    def custom_sort_key(file_name):
        sort_order = [
            'bau', 'no_cons', 'AAC_90', 'AAC_80', 'AAC_70', 
            'AAC_60', 'AAC_50', 'AAC_40', 'AAC_30', 
            'AAC_20', 'AAC_10'
        ]
        for idx, keyword in enumerate(sort_order):
            if keyword in file_name:
                return idx
        return len(sort_order)

    # Helper function to load and sort files
    def load_and_sort_files(folder_path, prefix, suffix):
        files = [
            f for f in os.listdir(folder_path)
            if f.startswith(prefix) and f.endswith(suffix)
        ]
        if not files:
            raise ValueError(f"No files found matching the prefix '{prefix}' in the folder '{folder_path}'.")
        files.sort(key=custom_sort_key)
        return [pd.read_csv(os.path.join(folder_path, file)) for file in files]

    # List of objective modes
    obj_modes = ['max_hv', 'min_ha', 'max_st', 'min_em']

    # Scenario colors for each objective mode
    scenario_colors = ['blue', 'green', 'orange', 'purple', 'red', 'brown', 'pink', 'gray', 'cyan', 'magenta']

    # Calculate global y-axis limits for Total Ecosystem Stocks
    global_min_stocks, global_max_stocks = float('inf'), float('-inf')
    for obj_mode in obj_modes:
        prefix = f"{case_study}_{obj_mode}"
        cbm_outputs_stocks = load_and_sort_files(folder_path_stocks, prefix, 'cbm_output_1.csv')
        for df in cbm_outputs_stocks:
            global_min_stocks = min(global_min_stocks, df['Total Ecosystem'].min())
            global_max_stocks = max(global_max_stocks, df['Total Ecosystem'].max())

    # Plot Total Ecosystem Stocks (One figure with 4 subplots)
    fig1, axes1 = plt.subplots(2, 2, figsize=(14, 10))
    for i, obj_mode in enumerate(obj_modes):
        prefix = f"{case_study}_{obj_mode}"
        cbm_outputs_stocks = load_and_sort_files(folder_path_stocks, prefix, 'cbm_output_1.csv')

        for df in cbm_outputs_stocks:
            df.index = range(1, len(df) + 1)

        ax = axes1[i // 2, i % 2]
        for j, df in enumerate(cbm_outputs_stocks):
            label = 'Baseline' if j == 0 else f'S{j-1}'
            color = 'black' if j == 0 else scenario_colors[j-1]
            df['Total Ecosystem'].plot(ax=ax, xlim=(0, n_steps), label=label, color=color)
        ax.set_ylim(global_min_stocks, global_max_stocks)  # Apply common y-axis limits
        ax.set_title(f'Total Ecosystem Stocks ({obj_mode})')
        ax.set_xlabel('Year')
        ax.set_ylabel('Total Ecosystem Stocks (tC)')
        ax.legend()
        ax.grid(True)

    fig1.tight_layout()
    fig1_path = os.path.join(fig_folder_path, f"{case_study}_total_ecosystem_stocks_comparisons.svg")
    fig1.savefig(fig1_path)

    # Calculate global y-axis limits for Net Emissions
    global_min_emissions, global_max_emissions = float('inf'), float('-inf')
    for obj_mode in obj_modes:
        prefix = f"{case_study}_{obj_mode}"
        cbm_outputs_emissions = load_and_sort_files(folder_path_emissions, prefix, 'cbm_output_2.csv')
        for df in cbm_outputs_emissions:
            global_min_emissions = min(global_min_emissions, df['Net emission'].min())
            global_max_emissions = max(global_max_emissions, df['Net emission'].max())

    # Plot Net Emissions (One figure with 4 subplots)
    fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
    for i, obj_mode in enumerate(obj_modes):
        prefix = f"{case_study}_{obj_mode}"
        cbm_outputs_emissions = load_and_sort_files(folder_path_emissions, prefix, 'cbm_output_2.csv')

        for df in cbm_outputs_emissions:
            df.index = range(1, len(df) + 1)

        ax = axes2[i // 2, i % 2]
        for j, df in enumerate(cbm_outputs_emissions):
            label = 'Baseline' if j == 0 else f'S{j-1}'
            color = 'black' if j == 0 else scenario_colors[j-1]
            df['Net emission'].plot(ax=ax, xlim=(0, n_steps), label=label, color=color)
        ax.axhline(0, color='red', linestyle='--', linewidth=1)
        ax.set_ylim(global_min_emissions, global_max_emissions)  # Apply common y-axis limits
        ax.set_title(f'Net Emissions ({obj_mode})')
        ax.set_xlabel('Year')
        ax.set_ylabel('Net Emissions (tCO₂e)')
        ax.legend()
        ax.grid(True)

    fig2.tight_layout()
    fig2_path = os.path.join(fig_folder_path, f"{case_study}_net_emissions_comparisons.svg")
    fig2.savefig(fig2_path)

    plt.show()

# Interactive function
def interactive_plot_emissions_and_stocks(case_study):
    folder_path_stocks = f'./plots/{case_study}/stocks'
    folder_path_emissions = f'./plots/{case_study}/emissions'
    plot_emissions_and_stocks(case_study, folder_path_stocks, folder_path_emissions)

# Widgets for user input
case_study_dropdown = Dropdown(
    options=['equitysilver', 'goldenbear', 'redchris'],  # Add more case study options
    value='equitysilver',
    description='Case Study:'
)

interact(interactive_plot_emissions_and_stocks, case_study=case_study_dropdown)


interactive(children=(Dropdown(description='Case Study:', options=('equitysilver', 'goldenbear', 'redchris'), …

<function __main__.interactive_plot_emissions_and_stocks(case_study)>